# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan Polars LazyFrame dan DuckDB. Isi sumber serta deskripsi dataset pada `data/README.md` sebelum menjalankan notebook ini.

In [1]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path('/home/jovyan/work')
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'wikipedia_id-raw.csv'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Unduh data ke data/raw/ lalu sesuaikan DATA_PATH.'
    )

print(f'Dataset: {DATA_PATH.name}')
print(f'Ukuran file: {DATA_PATH.stat().st_size / 1_000_000:.1f} MB')

Dataset: wikipedia_id-raw.csv
Ukuran file: 3010.1 MB


## Profil Polars

`scan_parquet` hanya membangun rencana query; file baru dibaca ketika `collect()` dipanggil.

In [2]:
lazy_data = pl.scan_csv(
    DATA_PATH,
    null_values='',
    schema_overrides={
        'id': pl.Int64,
        'parent_id': pl.Float64,
        'timestamp': pl.String,
        'hash': pl.String,
        'title': pl.String,
        'text': pl.String,
    },
)
print(lazy_data.collect_schema())

ringkasan = lazy_data.select([
    pl.len().alias('jumlah_baris'),
    pl.all().null_count().name.suffix('_null'),
]).collect(engine='streaming')

# Pemeriksaan awal; tidak mengubah atau menghapus record pada dataset mentah.
waktu = pl.col('timestamp').str.to_datetime(
    format='%Y-%m-%dT%H:%M:%SZ', time_zone='UTC', strict=False
)
panjang_teks = pl.col('text').fill_null('').str.len_chars()
profil_awal = lazy_data.select([
    waktu.min().alias('waktu_terawal'),
    waktu.max().alias('waktu_terakhir'),
    (waktu.is_null().sum() - pl.col('timestamp').null_count()).alias('timestamp_tidak_valid'),
    (pl.col('id').count() - pl.col('id').drop_nulls().n_unique()).alias('duplikat_id_tambahan'),
    (pl.col('title').count() - pl.col('title').drop_nulls().n_unique()).alias('duplikat_judul_tambahan'),
    (pl.col('hash').count() - pl.col('hash').drop_nulls().n_unique()).alias('duplikat_hash_tambahan'),
    pl.col('title').str.strip_chars().eq('').fill_null(False).sum().alias('judul_hanya_spasi'),
    panjang_teks.min().alias('panjang_teks_min'),
    panjang_teks.median().alias('panjang_teks_median'),
    panjang_teks.max().alias('panjang_teks_max'),
    pl.col('text').str.contains(r'\[\[|\{\{').fill_null(False).sum().alias('teks_dengan_markup_wiki'),
]).collect(engine='streaming')

for nama, nilai in profil_awal.row(0, named=True).items():
    print(f'{nama}: {nilai}')

ringkasan

Schema({'id': Int64, 'parent_id': Float64, 'timestamp': String, 'hash': String, 'title': String, 'text': String})
waktu_terawal: 2004-04-19 13:00:14+00:00
waktu_terakhir: 2022-11-20 10:49:11+00:00
timestamp_tidak_valid: 0
duplikat_id_tambahan: 0
duplikat_judul_tambahan: 0
duplikat_hash_tambahan: 264115
judul_hanya_spasi: 0
panjang_teks_min: 0
panjang_teks_median: 127.0
panjang_teks_max: 1887100
teks_dengan_markup_wiki: 1542166


jumlah_baris,id_null,parent_id_null,timestamp_null,hash_null,title_null,text_null
u32,u32,u32,u32,u32,u32,u32
1550872,0,602033,0,0,1,993


In [3]:
contoh_data = lazy_data.head(10).collect()
contoh_data

id,parent_id,timestamp,hash,title,text
i64,f64,str,str,str,str
1,1.9472643e7,"""2022-04-24T00:35:51Z""","""oo2rvi3f9p4u9czu3opfvbsl0jp9ao…","""Asam deoksiribonukleat""","""[[Berkas:DNA Structure+Key+Lab…"
2,1.7982206e7,"""2021-02-15T10:17:21Z""","""49bjs0f6ys3j5ymkyg8c41ino347yk…","""Asam Deoksiribosanukleat""","""#ALIH [[Asam deoksiribonukleat…"
3,1.9395623e7,"""2022-05-05T11:26:32Z""","""lq3uz0weougma29prpmkm1l5at48s4…","""Anwar Sadat""","""{{Infobox officeholder | nativ…"
4,1.4292725e7,"""2021-07-05T02:41:20Z""","""3h1z248a5ic1u3odrse27wg0o0sllj…","""Azhar Mansor""","""[[Berkas:Azhar_Mansor_Jalur_Ge…"
5,2.1368465e7,"""2022-09-26T04:25:33Z""","""ololfquhgvu4wzszfu1yaqo3amb8xn…","""Arkeologi""","""{{Terjemah|Inggris}} {{Ilmu}} …"
6,2.1768805e7,"""2022-11-02T02:33:49Z""","""mkxdb4xfmgse21h6lvtoflnvwlepdb…","""Antropologi""","""{{antropologi}} {{Zoologi}} ''…"
7,1.514347e7,"""2021-10-11T13:49:00Z""","""q250m0j8cxxnornkw4s88o8jvr4rwh…","""Analisis leksikal""","""'''Analisis leksikal''' (bahas…"
8,2.1997603e7,"""2022-11-16T04:21:39Z""","""9n8fb5ztwyy71xe1emj3glw988glko…","""Bahasa Indonesia""","""{{Infobox Bahasa |name={{pagen…"
9,2.1831638e7,"""2022-11-11T11:21:27Z""","""q6z7ul9flklj70wim4f9vy3ijmytnn…","""Biologi""","""{{multiple image|perrow=2|tota…"


## Profil DuckDB

DuckDB membaca file Parquet langsung dari disk dan menghasilkan ringkasan statistik setiap kolom.

In [4]:
connection = duckdb.connect()
try:
    profil_duckdb = connection.execute(
        """
        SUMMARIZE SELECT * FROM read_csv(
            ?, header = true, delim = ',', quote = '"', escape = '"', nullstr = '',
            columns = {
                'id': 'BIGINT',
                'parent_id': 'DOUBLE',
                'timestamp': 'VARCHAR',
                'hash': 'VARCHAR',
                'title': 'VARCHAR',
                'text': 'VARCHAR'
            },
            max_line_size = 33554432
        )
        """,
        [str(DATA_PATH)],
    ).pl()
finally:
    connection.close()

profil_duckdb

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""id""","""BIGINT""","""1""","""3768441""",1985843,"""1827983.623075921""","""1124996.968842503""","""778775""","""1937394""","""2798800""",1550872,0.00
"""parent_id""","""DOUBLE""","""2316.0""","""22030575.0""",1121672,"""15271049.586341834""","""5669601.806865781""","""12295647.668288881""","""16990368.258274294""","""20084634.48603593""",1550872,38.82
"""timestamp""","""VARCHAR""","""2004-04-19T13:00:14Z""","""2022-11-20T10:49:11Z""",1339674,null,null,null,null,null,1550872,0.00
"""hash""","""VARCHAR""","""0000ymuyndr6l1q4hc4kfdhgnrcpbi…","""twj48pdguc12bpfm814uum83mdnrl0…",1355277,null,null,null,null,null,1550872,0.00
"""title""","""VARCHAR""","""!""","""🪙""",1551409,null,null,null,null,null,1550872,0.00
"""text""","""VARCHAR""","""! colspan=""2"" style=""color:#ff…","""🚉 <noinclude>[[Kategori:Jalan …",1527130,null,null,null,null,null,1550872,0.06


## Catatan Milestone 1

Tulis temuan awal di bawah ini: ukuran data, jumlah baris, tipe kolom, kolom dengan null terbanyak, serta anomali yang perlu ditangani pada Milestone 2.

Temuan awal
- Ukuran file aktual: 3.010.074.263 byte (3010.07 MB atau 3.01 GB, satuan desimal). Dataset memenuhi syarat minimal 500 MB atau lebih dari 1 juta baris.
- Jumlah record CSV: 1.550.872 baris, dengan 6 kolom. Baris baru di dalam isi artikel tidak dihitung sebagai record tambahan.
- Rentang timestamp yang valid: 2004-04-19 13:00:14 UTC sampai 2022-11-20 10:49:11 UTC. Ini merupakan rentang timestamp pada record, bukan tanggal unduh atau rentang riwayat lengkap artikel.
- Tipe kolom yang digunakan dan jumlah nilai kosong (field CSV kosong dibaca sebagai null):

| Kolom | Tipe Polars | Jumlah null | Persentase null |
|---|---|---:|---:|
| `id` | Int64 | 0 | 0.0000% |
| `parent_id` | Float64 | 602.033 | 38.8190% |
| `timestamp` | String | 0 | 0.0000% |
| `hash` | String | 0 | 0.0000% |
| `title` | String | 1 | 0.0001% |
| `text` | String | 993 | 0.0640% |

Anomali dan rencana Milestone 2
- Kolom dengan null terbanyak: parent_id, sebanyak 602.033 record. Periksa penyebabnya sebelum menentukan apakah perlu diisi, dipertahankan, atau dikeluarkan dari analisis tertentu.
- Duplikat tambahan pada id: 0; pada title non-null: 0. Angka ini menghitung jumlah record non-null dikurangi jumlah nilai unik, bukan duplikasi identik seluruh baris. Verifikasi kunci dan makna revisi sebelum melakukan deduplikasi.
- Duplikat tambahan pada hash non-null: 264.115. Hash berulang belum membuktikan seluruh baris identik; periksa kesamaan konten dan jenis halaman sebelum menghapus data.
- Timestamp non-kosong yang gagal diparsing dengan format ISO UTC: 0.
- 1.542.166 record teks memuat [[ atau {{, sehingga masih mengandung penanda markup Wikipedia. Pembersihan markup perlu disesuaikan dengan tujuan analisis teks.
- Panjang teks minimum, median, dan maksimum: 0, 127, dan 1.887.100 karakter, dengan null diperlakukan sebagai teks kosong hanya untuk metrik panjang. Artikel panjang perlu ditinjau sebagai kandidat outlier; panjang teks saja belum cukup untuk menghapus record.
- parent_id tersimpan sebagai bilangan pecahan pada format CSV. Pemeriksaan menemukan 0 nilai non-kosong dengan bagian pecahan bukan nol; evaluasi konversi ke integer nullable pada tahap cleaning.